# Twitch Emote Prediction — VOD Scraper

Processes a Twitch VOD to produce a labelled training dataset. Each sample is a triplet of:
- **Video**: 64 evenly-sampled frames, center-cropped and resized to 256×256 (MP4)
- **Audio**: the corresponding audio segment (MP3)
- **Target**: a normalized emote-frequency vector representing chat hype during that window (JSON)

### Cells

- **Cell 1 — Configuration**: Set `VOD_ID`, `CLIP_DURATION`, `MIN_EMOTES`, and other parameters before running.
- **Cell 2 — Helpers & Pipeline**: Downloads the VOD via yt-dlp; scrapes full chat history through Twitch's GQL API with cursor-based pagination and resume support; fetches third-party emotes (BTTV, FFZ, 7TV); processes each high-activity window into a video/audio/target triplet.
- **Cell 3 — Export**: Zips the completed dataset and saves it to the Google Drive folder specified by `DRIVE_OUTPUT_FOLDER`.

### Prerequisites

Credentials are read from Colab's secret store (`Tools → Secrets`). Add these before running:

| Secret | Value |
|---|---|
| `CLIENT_ID` | Twitch app client ID |
| `AUTHORIZATION` | OAuth token (`OAuth …`) |
| `CLIENT_INTEGRITY` | Twitch client integrity token |
| `DEVICE_ID` | Twitch device ID |

In [ ]:
import os
import subprocess
import json
from pathlib import Path
import requests
import cv2
import numpy as np
import tqdm

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
VOD_ID = "1594760722"  # ← change this to process a different VOD
OUTPUT_FOLDER = f"./dataset_{VOD_ID}"
CLIP_DURATION = 15     # seconds per clip window
FRAMES_PER_CLIP = 64   # frames sampled evenly across each clip
TARGET_RES = 256       # output frame resolution (square)
MIN_EMOTES = 5         # minimum emote count in a window to be included
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/twitch_emote_data"  # ← set your Drive folder path here
VOD_URL = f"https://www.twitch.tv/videos/{VOD_ID}" 

## Dependencies

In [ ]:
!pip install -q yt-dlp curl-cffi

## VOD Scraping Pipeline

### Design Decisions

**Emote density as the hype signal** — Emotes are intentional emotional reactions; chat floods with `PogChamp`, `KEKW`, or `monkaS` when something exciting happens. Raw message count is a weaker signal since people type just as much during dull moments. Emote frequency requires no human annotation and naturally captures community-specific excitement.

**Third-party emotes (BTTV, FFZ, 7TV)** — Twitch's native emote set covers only a fraction of what any given community actually uses. The most expressive, community-specific emotes live on these platforms, so excluding them would leave the majority of the hype signal undetected.

**15-second clip windows** — Long enough to contain a complete moment, short enough that one event dominates the window. Longer windows blur distinct moments together; shorter ones often clip the reaction before it fully lands in chat.

**64 frames per clip** — Fixed-size input is required by V-JEPA 2. 64 is a power of 2 (efficient for batching), and at 15 seconds yields roughly 4fps — enough temporal coverage to represent motion without excessive storage cost.

**Center crop** — Game streams are typically composed with the action centered. Center cropping is simple, deterministic, and requires no additional model. Face detection was considered but adds latency and fails on gameplay-heavy content where no face is present.

**256×256 resolution** — Balances visual detail against compute and storage cost. Standard for many vision models and keeps per-clip file sizes manageable at scale.

**Normalized emote vector** — Raw emote counts vary with overall chat volume. Normalizing to a probability distribution makes windows comparable regardless of how active the chat is, so the model learns which emotes dominate a moment rather than how loud the chat was.

**Twitch GQL API for chat** — Twitch's public API does not expose full VOD chat history. The GQL endpoint is what Twitch's own web client uses internally — it returns complete message history with native emote metadata, supports cursor-based pagination, and is reliable enough for long VODs.

In [ ]:
import os
import json
import subprocess
import time
from pathlib import Path

import cv2
import numpy as np
import tqdm
import yt_dlp
from curl_cffi import requests
from google.colab import userdata



MY_CLIENT_ID = userdata.get('CLIENT_ID')
MY_OAUTH = userdata.get('AUTHORIZATION')
MY_INTEGRITY = userdata.get('CLIENT_INTEGRITY')
DEVICE_ID = userdata.get('DEVICE_ID')

_missing = [name for name, val in {
    'CLIENT_ID': MY_CLIENT_ID,
    'AUTHORIZATION': MY_OAUTH,
    'CLIENT_INTEGRITY': MY_INTEGRITY,
    'DEVICE_ID': DEVICE_ID,
}.items() if not val]
if _missing:
    raise RuntimeError(
        f"Missing Colab secret(s): {', '.join(_missing)}\n"
        "Go to Tools → Secrets and add them before running."
    )

# ==========================================
# 1. HELPERS (Global Scope)
# ==========================================



def get_third_party_emotes(channel_id=None):
    """Fetches Global and Channel-specific emotes from BTTV, FFZ, and 7TV."""
    emotes = set()

    # --- 1. BTTV (BetterTTV) ---
    try:
        resp = requests.get("https://api.betterttv.net/3/cached/emotes/global", timeout=5)
        if resp.status_code == 200:
            for e in resp.json(): emotes.add(e['code'])

        if channel_id:
            resp = requests.get(f"https://api.betterttv.net/3/cached/users/twitch/{channel_id}", timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                for e in data.get('channelEmotes', []) + data.get('sharedEmotes', []):
                    emotes.add(e['code'])
    except Exception as e: print(f"⚠️ BTTV request failed: {e}")

    # --- 2. FFZ (FrankerFaceZ) ---
    try:
        resp = requests.get("https://api.frankerfacez.com/v1/set/global", timeout=5)
        if resp.status_code == 200:
            for set_data in resp.json().get('sets', {}).values():
                for e in set_data.get('emoticons', []): emotes.add(e['name'])

        if channel_id:
            resp = requests.get(f"https://api.frankerfacez.com/v1/room/id/{channel_id}", timeout=5)
            if resp.status_code == 200:
                for set_data in resp.json().get('sets', {}).values():
                    for e in set_data.get('emoticons', []): emotes.add(e['name'])
    except Exception as e: print(f"⚠️ FFZ request failed: {e}")

    # --- 3. 7TV ---
    try:
        resp = requests.get("https://7tv.io/v3/emote-sets/global", timeout=5)
        if resp.status_code == 200:
            for e in resp.json().get('emotes', []): emotes.add(e['name'])

        if channel_id:
            resp = requests.get(f"https://7tv.io/v3/users/twitch/{channel_id}", timeout=5)
            if resp.status_code == 200:
                for e in resp.json().get('emote_set', {}).get('emotes', []):
                    emotes.add(e['name'])
    except Exception as e: print(f"⚠️ 7TV request failed: {e}")

    print(f"Loaded {len(emotes)} third-party emotes (BTTV + FFZ + 7TV).")
    return emotes

def get_vod_metadata(vod_id):
    ydl_opts = {'quiet': True, 'simulate': True}

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(f"https://www.twitch.tv/videos/{vod_id}", download=False)

            duration = info.get('duration')
            streamer_name = info.get('uploader_id') or info.get('uploader')

            if duration:
                print(f"VOD duration: {duration}s")
            else:
                print("⚠️ Could not determine VOD duration from metadata.")

            if streamer_name:
                resp = requests.get(f"https://decapi.me/twitch/id/{streamer_name}", timeout=5)
                if resp.status_code == 200 and "User not found" not in resp.text:
                    channel_id = resp.text.strip()
                    if channel_id and duration:
                        return channel_id, duration
                    if not duration:
                        print("⚠️ VOD duration could not be determined automatically.")
                else:
                    print(f"⚠️ DecAPI could not resolve ID for {streamer_name}.")
            else:
                print("⚠️ Streamer name not found in VOD metadata.")

    except Exception as e:
        print(f"⚠️ Error during metadata fetch: {e}")

    # The Ultimate Failsafe
    print("\n❌ Automation failed. Manual input required.")
    manual_id = input("   👉 Go to https://decapi.me/twitch/id/STREAMER_NAME in your browser and paste the number here: ")
    manual_duration = int(input("   👉 Enter the VOD duration in seconds (e.g. 16927): "))
    return str(manual_id).strip(), manual_duration

def fetch_chat_oauth(vod_id, max_vod_seconds):
    url = "https://gql.twitch.tv/gql"
    headers = {
        "Client-Id": MY_CLIENT_ID,
        "Authorization": MY_OAUTH,
        "Client-Integrity": MY_INTEGRITY,
        "X-Device-Id": DEVICE_ID,
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
        "Content-Type": "application/json",
    }

    messages = []
    seen_messages = set()
    current_offset = 0

    # --- RESUME LOGIC ---
    save_file = f"chat_cache_{vod_id}.json"
    if os.path.exists(save_file):
        try:
            with open(save_file, "r", encoding="utf-8") as f:
                messages = json.load(f)
            if messages:
                current_offset = int(messages[-1]['time_in_seconds'])
                seen_messages = {m['id'] for m in messages if 'id' in m}
                print(f"Resuming from {current_offset}s ({len(messages)} messages cached).")
        except Exception as e:
            print(f"⚠️ Could not load save file: {e}. Starting from scratch.")
            messages = []
            current_offset = 0
    else:
        print("Fetching chat...")

    requests_made = 0
    MAX_RETRIES = 3
    retry_count = 0
    cursor = None

    with tqdm.tqdm(total=max_vod_seconds, initial=int(current_offset), desc="Scraping Timeline") as pbar:
        while current_offset <= max_vod_seconds:

            variables = {
                "videoID": vod_id,
                "contentOffsetSeconds": int(current_offset)
            }

            if cursor:
                variables["cursor"] = cursor

            query = [{
                "operationName": "VideoCommentsByOffsetOrCursor",
                "variables": variables,
                "extensions": {
                    "persistedQuery": {
                        "version": 1,
                        "sha256Hash": "b70a3591ff0f4e0313d126c6a1502d79a1c02baebb288227c582044aa76adf6a"
                    }
                }
            }]

            try:
                # timeout=10 prevents indefinite hangs
                resp = requests.post(url, json=query, headers=headers, timeout=10)
                requests_made += 1

                if resp.status_code != 200:
                    print(f"\n❌ HTTP {resp.status_code} at {current_offset}s. Check that your client integrity token is valid.")
                    break

                res_data = resp.json()[0]

                if 'errors' in res_data:
                    error_msg = res_data['errors'][0]['message']
                    if "service error" in error_msg.lower() or "timeout" in error_msg.lower():
                        if retry_count < MAX_RETRIES:
                            print(f"\n⚠️ Twitch service error at {current_offset}s. Retrying ({retry_count + 1}/{MAX_RETRIES})...")
                            retry_count += 1
                            time.sleep(1.5)
                            continue
                        else:
                            print(f"Repeated failures at {current_offset}s, skipping 1s and continuing.")
                            current_offset += 1
                            cursor = None
                            pbar.update(1)
                            retry_count = 0
                            continue
                    else:
                        print(f"\n❌ Unrecoverable Twitch error at {current_offset}s: {error_msg}")
                        break

                retry_count = 0

                comments_data = res_data.get('data', {}).get('video', {}).get('comments', {})
                edges = comments_data.get('edges', [])

                page_info = comments_data.get('pageInfo', {})
                has_next_page = page_info.get('hasNextPage', False)

                if not edges:
                    current_offset += 1
                    cursor = None
                    pbar.update(1)
                    continue

                for edge in edges:
                    node = edge['node']
                    msg_id = node.get('id')

                    if msg_id in seen_messages:
                        continue
                    seen_messages.add(msg_id)

                    msg_text = ""
                    emotes = []

                    if node['message'].get('fragments'):
                        for f in node['message']['fragments']:
                            frag_text = f.get('text', '')
                            msg_text += frag_text

                            if f.get('emote'):
                                emotes.append({
                                    "name": frag_text.strip(),
                                    "id": f['emote'].get('emoteID', 'unknown')
                                })

                    messages.append({
                        'id': msg_id,
                        'time_in_seconds': node.get('contentOffsetSeconds', current_offset),
                        'message': msg_text.strip(),
                        'emotes': emotes
                    })

                # --- PAGINATION LOGIC ---
                # 1. Figure out what timestamp the last message in this chunk had
                last_msg_time = edges[-1]['node'].get('contentOffsetSeconds', current_offset)
                next_offset = int(last_msg_time)

                # 2. If we moved forward in time, update the progress bar!
                progress_made = next_offset - int(current_offset)
                if progress_made > 0:
                    pbar.update(progress_made)
                    current_offset = next_offset

                # 3. Handle the cursor for the next loop
                if has_next_page:
                    cursor = edges[-1].get('cursor')
                else:
                    cursor = None
                    # Failsafe: if there's no next page and we are stuck on the same second, force move forward
                    if progress_made <= 0:
                        current_offset += 1
                        pbar.update(1)

                time.sleep(0.2)

                # --- PERIODIC SAVE ---
                if requests_made % 50 == 0:
                    with open(save_file, "w", encoding="utf-8") as f:
                        json.dump(messages, f)

            except Exception as e:
                if retry_count < MAX_RETRIES:
                    print(f"⚠️ Network error at {current_offset}s: {e}. Retrying ({retry_count + 1}/{MAX_RETRIES})...")
                    retry_count += 1
                    time.sleep(3) # Wait a bit longer for network to stabilize
                    continue
                else:
                    print(f"\n❌ Network error unrecoverable after {MAX_RETRIES} retries. Stopping.")
                    break

    # --- FINAL SAVE ON EXIT ---
    try:
        with open(save_file, "w", encoding="utf-8") as f:
            json.dump(messages, f)
    except Exception as e:
        print(f"⚠️ Failed to save final output: {e}")

    return messages
# ==========================================
# 2. MAIN PIPELINE
# ==========================================

def run_pipeline():
    # Setup folders
    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    video_dir = Path(OUTPUT_FOLDER) / "video"
    audio_dir = Path(OUTPUT_FOLDER) / "audio"
    target_dir = Path(OUTPUT_FOLDER) / "target"
    for d in [video_dir, audio_dir, target_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # --- STEP A: DOWNLOAD VIDEO ---
    print(f"\nDownloading VOD {VOD_ID}...")
    video_path = f"{OUTPUT_FOLDER}/full_vod.mp4"

    if not os.path.exists(video_path):
        ydl_opts = {
            'format': 'best[height<=360]/best',
            'outtmpl': video_path,
            'merge_output_format': 'mp4',
            'concurrent_fragment_downloads': 5, # Safe threading
            'quiet': False
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(VOD_URL, download=False)
                ydl.download([VOD_URL])
        except Exception as e:
            print(f"\n❌ Video download failed: {e}")
            return
    else:
        print("Video already downloaded, skipping.")

    # --- STEP B: METADATA & CHAT ---
    streamer_id, vod_duration = get_vod_metadata(VOD_ID)

    print(f"\nFetching chat for VOD {VOD_ID}...")
    messages = fetch_chat_oauth(VOD_ID, vod_duration)

    if not messages:
        print("❌ No chat messages retrieved. Verify your OAuth token is valid.")
        return
    else:
        print(f"Retrieved {len(messages)} messages.")


    third_party_emotes = get_third_party_emotes(channel_id=streamer_id)

    buckets = {}

    for msg in messages:
        offset = msg.get('time_in_seconds')
        if offset is None: continue

        # Calculate which 10-second (or whatever CLIP_DURATION is) chunk this belongs to
        window_start = (int(offset) // CLIP_DURATION) * CLIP_DURATION
        if window_start not in buckets:
            buckets[window_start] = {}

        # Track volume
        buckets[window_start]['_total_messages'] = buckets[window_start].get('_total_messages', 0) + 1

        counted_native = set() # Track to prevent double counting

        # 1. Count Native Emotes
        if msg.get('emotes'):
            for e in msg['emotes']:
                ename = e.get('name', 'unknown').strip()

                buckets[window_start][ename] = buckets[window_start].get(ename, 0) + 1
                counted_native.add(ename)

        # 2. Count 3rd-Party Emotes from Plain Text
        words = msg.get('message', '').split()
        for word in words:
            # We check the third_party_emotes list now!
            if word in third_party_emotes and word not in counted_native:
                buckets[window_start][word] = buckets[window_start].get(word, 0) + 1

    for threshold in [5, 10, 15, 20, 30, 50]:
      count = len([t for t, d in buckets.items() if sum(v for k, v in d.items() if k != '_total_messages') >= threshold])
      print(f"Threshold {threshold}: {count} clips found")

    # The new filtering logic: only keep windows where the sum of actual emotes meets the threshold
    interesting_starts = [
        t for t, dist in buckets.items()
        if sum(v for k, v in dist.items() if k != '_total_messages') >= MIN_EMOTES
    ]
    interesting_starts.sort()

    if not interesting_starts:
        print(f"❌ No segments found with >= {MIN_EMOTES} emotes.")
        return

    # --- Emote Mapping ---
    all_unique_emotes = set()

    for t, dist in buckets.items():
        for emote_key in dist.keys():
            # exclude volume tracker key from emote features
            if emote_key != '_total_messages':
                all_unique_emotes.add(emote_key)

    # Sort alphabetically so our ML vector indexes never shift around
    sorted_emotes = sorted(list(all_unique_emotes))
    emote_to_index = {emote: i for i, emote in enumerate(sorted_emotes)}

    emote_mapping_path = Path(OUTPUT_FOLDER) / 'emote_mapping.json'
    with open(emote_mapping_path, 'w', encoding='utf-8') as f:
        json.dump(emote_to_index, f, indent=4)

    print(f"Emote vocabulary: {len(sorted_emotes)} unique emotes.")

    # --- STEP C: PROCESS & SAVE ---
    print(f"\nProcessing {len(interesting_starts)} clips...")
    try:
        cap = cv2.VideoCapture(video_path)
        original_fps = cap.get(cv2.CAP_PROP_FPS)

        # Pre-calculate cropping
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        s = min(orig_w, orig_h)
        x, y = (orig_w - s) // 2, (orig_h - s) // 2

        out_fps = FRAMES_PER_CLIP / CLIP_DURATION

        for i, start_time in enumerate(tqdm.tqdm(interesting_starts)):
            clip_id = f"clip_{VOD_ID}_{i:05d}"
            vid_f, aud_f, tgt_f = video_dir/f"{clip_id}.mp4", audio_dir/f"{clip_id}.mp3", target_dir/f"{clip_id}.json"

            # 1. Target Vector
            if not tgt_f.exists():
                emote_vector = np.zeros(len(all_unique_emotes))
                for emote, count in buckets[start_time].items():
                  if emote != '_total_messages':
                      emote_vector[emote_to_index[emote]] = count

                if emote_vector.sum() > 0:
                  emote_vector /= emote_vector.sum()

                with open(tgt_f, 'w') as f:
                  json.dump(emote_vector.tolist(), f)

            # 2. Video Crop & Resample
            if not vid_f.exists():
                start_frame = int(start_time * original_fps)
                end_frame = int((start_time + CLIP_DURATION) * original_fps)
                total_frames = end_frame - start_frame

                sample_indices = set(np.linspace(0, total_frames, FRAMES_PER_CLIP, endpoint=False, dtype=int))

                cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
                frames_to_write = []

                for rel_idx in range(total_frames):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if rel_idx in sample_indices:
                        crop = frame[y:y+s, x:x+s]
                        resized = cv2.resize(crop, (TARGET_RES, TARGET_RES))
                        frames_to_write.append(resized)

                # enforce exactly FRAMES_PER_CLIP frames
                if len(frames_to_write) == 0:
                    # Failsafe: Completely unreadable segment
                    frames_to_write = [np.zeros((TARGET_RES, TARGET_RES, 3), dtype=np.uint8)] * FRAMES_PER_CLIP
                elif len(frames_to_write) < FRAMES_PER_CLIP:
                    # Failsafe: Pad short clips by repeating the last valid frame
                    last_frame = frames_to_write[-1]
                    padding = [last_frame] * (FRAMES_PER_CLIP - len(frames_to_write))
                    frames_to_write.extend(padding)
                elif len(frames_to_write) > FRAMES_PER_CLIP:
                    # Failsafe: Truncate if too long
                    frames_to_write = frames_to_write[:FRAMES_PER_CLIP]

                # --- WRITE TO DISK ---
                writer = cv2.VideoWriter(str(vid_f), cv2.VideoWriter_fourcc(*'mp4v'), out_fps, (TARGET_RES, TARGET_RES))
                for f in frames_to_write:
                    writer.write(f)
                writer.release()


            # 3. Audio Extraction
            if not aud_f.exists():
                try:
                    subprocess.run([
                        "ffmpeg", "-y", "-ss", str(start_time), "-t", str(CLIP_DURATION),
                        "-i", video_path, "-vn", "-acodec", "libmp3lame", str(aud_f)
                    ], check=True, capture_output=True)
                except subprocess.CalledProcessError as e:
                    print(f"⚠️ Audio extraction failed for clip at {start_time}s: {e}")

    finally:
        cap.release()
    print(f"\nDone. Dataset saved to {OUTPUT_FOLDER}.")


run_pipeline()

## Export

Zips the completed dataset and saves it to Google Drive. The archive contains three subdirectories (`video/`, `audio/`, `target/`) and `emote_mapping.json`. The full VOD file is excluded to keep the archive size manageable.

In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create the destination folder if it doesn't exist
os.makedirs(DRIVE_OUTPUT_FOLDER, exist_ok=True)

# Zip only the dataset triplets — exclude the full VOD and any cache files
import zipfile
zip_filename = f"dataset_{VOD_ID}_clips"
zip_path = f"{zip_filename}.zip"
print(f"Compressing '{OUTPUT_FOLDER}'...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for subdir in ['video', 'audio', 'target']:
        subdir_path = os.path.join(OUTPUT_FOLDER, subdir)
        if os.path.exists(subdir_path):
            for root, _, files in os.walk(subdir_path):
                for file in files:
                    filepath = os.path.join(root, file)
                    zf.write(filepath, os.path.relpath(filepath, os.path.dirname(OUTPUT_FOLDER)))
    mapping = os.path.join(OUTPUT_FOLDER, 'emote_mapping.json')
    if os.path.exists(mapping):
        zf.write(mapping, os.path.relpath(mapping, os.path.dirname(OUTPUT_FOLDER)))

# Copy to Drive and remove local zip to save Colab disk space
dest = os.path.join(DRIVE_OUTPUT_FOLDER, zip_path)
shutil.copy(zip_path, dest)
os.remove(zip_path)
print(f"Saved to Drive: {dest}")